In [24]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))
from scripts.data_loader import load_r2_data

## Load Data

# Separations Analysis
Explores federal employee separation events from Jan 2025 – Feb 2026, with a focus on identifying DOGE-driven involuntary separations by agency and employee profile.

In [2]:
# Load all separations data from R2 (2024-2026)
df = load_r2_data("separations")

/Users/destin/Projects/federal-doge-study/.venv/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


  loading separations/2025/separations_202501.txt …
  loading separations/2025/separations_202502.txt …
  loading separations/2025/separations_202503.txt …
  loading separations/2025/separations_202504.txt …
  loading separations/2025/separations_202505.txt …
  loading separations/2025/separations_202506.txt …
  loading separations/2025/separations_202507.txt …
  loading separations/2025/separations_202508.txt …
  loading separations/2025/separations_202509.txt …
  loading separations/2025/separations_202510.txt …
  loading separations/2025/separations_202511.txt …
  loading separations/2025/separations_202512.txt …
  loading separations/2026/separations_202601.txt …
  loading separations/2026/separations_202602.txt …


## Overview
High-level counts to understand the shape of the data — total separations by type, time, and agency.

In [25]:
df['separation_category'].value_counts()

separation_category
QUIT                                  175823
RETIREMENT - VOLUNTARY                124617
TERMINATION (EXPIRED APPT/OTHER)       41374
RETIREMENT - EARLY OUT                 28093
OTHER SEPARATION                       18852
TRANSFER OUT - INDIVIDUAL TRANSFER     12808
REDUCTION IN FORCE (RIF)               10629
RETIREMENT - OTHER                      5553
TRANSFER OUT - MASS TRANSFER              90
Name: count, dtype: int64

In [26]:
sep_time = df.groupby(["year", "month"]).size()
sep_time

year  month
2025  1         22290
      2         15635
      3         22114
      4         24176
      5         26736
      6         20116
      7         22967
      8         18529
      9        123050
      10        12648
      11        19955
      12        30823
2026  1         39630
      2         19170
dtype: int64

In [27]:
sep_agency = df.groupby("agency_subelement").size().sort_values(ascending=False)
sep_agency.head(10)

agency_subelement
VETERANS HEALTH ADMINISTRATION             50779
INTERNAL REVENUE SERVICE                   33237
FOREST SERVICE                             10416
AIR FORCE MATERIEL COMMAND                  9344
NATIONAL PARK SERVICE                       8968
TRANSPORTATION SECURITY ADMINISTRATION      8578
SOCIAL SECURITY ADMINISTRATION              8147
MILITARY TREATMENT FACILITIES UNDER DHA     6530
FEDERAL AVIATION ADMINISTRATION             6382
U.S. ARMY CORPS OF ENGINEERS                5876
dtype: int64

In [28]:
share_by_age = df["age_bracket"].value_counts(normalize=True)
share_by_age


age_bracket
60-64           0.155337
65 OR MORE      0.139441
55-59           0.133408
50-54           0.094197
35-39           0.092449
40-44           0.089563
30-34           0.088525
25-29           0.079715
45-49           0.077379
20-24           0.047286
LESS THAN 20    0.002700
Name: proportion, dtype: float64

In [29]:
share_by_tenure = df["tenure_code"].value_counts(normalize=True)
share_by_tenure

tenure_code
1    0.555941
2    0.174483
1    0.077388
0    0.076272
3    0.054390
2    0.027818
0    0.019341
3    0.014336
*    0.000031
Name: proportion, dtype: float64

## Voluntary vs Involuntary Classification

DOGE signal: are involuntary separations (RIF, terminations) rising?

In [30]:
import pandas as pd

# Classify each separation
INVOLUNTARY = {"SH", "SJ"}   # RIF, Termination
VOLUNTARY   = {"SC", "SD", "SE", "SG", "SA", "SB"}  # Quit, Retirements, Transfers

def classify(code):
    if code in INVOLUNTARY:
        return "involuntary"
    elif code in VOLUNTARY:
        return "voluntary"
    else:
        return "other"

df["sep_class"] = df["separation_category_code"].apply(classify)

# Period label — DOGE executive orders began Jan 20 2025
# Jan 2025 = transition month; Feb 2025+ = DOGE in effect; 2026 = acceleration
def period(row):
    if row["year"] == 2025 and row["month"] == 1:
        return "Jan-2025 (transition)"
    elif row["year"] == 2025:
        return "Feb-Dec 2025 (DOGE year 1)"
    else:
        return "2026 (DOGE year 2)"

df["period"] = df.apply(period, axis=1)

df["sep_class"].value_counts()

sep_class
voluntary      346984
involuntary     52003
other           18852
Name: count, dtype: int64

## Monthly Trend by Separation Type

In [31]:
# Monthly counts by separation category code
monthly = (
    df.groupby(["year", "month", "separation_category_code", "separation_category"])
    .size()
    .reset_index(name="count")
)
monthly["ym"] = monthly["year"].astype(str) + "-" + monthly["month"].astype(str).str.zfill(2)
monthly.sort_values("ym", inplace=True)

# Pivot: rows=month, cols=separation type
pivot = monthly.pivot_table(index="ym", columns="separation_category", values="count", fill_value=0)
pivot

separation_category,OTHER SEPARATION,QUIT,REDUCTION IN FORCE (RIF),RETIREMENT - EARLY OUT,RETIREMENT - OTHER,RETIREMENT - VOLUNTARY,TERMINATION (EXPIRED APPT/OTHER),TRANSFER OUT - INDIVIDUAL TRANSFER,TRANSFER OUT - MASS TRANSFER
ym,,,,,,,,,
2025-01,1371.0,9141.0,1.0,24.0,303.0,6544.0,1856.0,3030.0,20.0
2025-02,1889.0,5777.0,1.0,83.0,279.0,4643.0,1728.0,1230.0,5.0
2025-03,1934.0,9672.0,24.0,713.0,318.0,6141.0,2921.0,390.0,1.0
2025-04,1424.0,8848.0,178.0,3147.0,269.0,6553.0,3284.0,470.0,3.0
2025-05,1498.0,10431.0,253.0,1102.0,378.0,7639.0,4761.0,671.0,3.0
2025-06,1254.0,8556.0,513.0,796.0,344.0,5064.0,3063.0,522.0,4.0
2025-07,1325.0,7824.0,5392.0,299.0,1139.0,3571.0,2866.0,544.0,7.0
2025-08,1254.0,8349.0,1380.0,407.0,566.0,3517.0,2402.0,651.0,3.0
2025-09,1338.0,69358.0,2389.0,12314.0,494.0,32407.0,3654.0,1091.0,5.0


In [32]:
# Voluntary vs involuntary totals by month
monthly_class = (
    df.groupby(["year", "month", "sep_class"])
    .size()
    .reset_index(name="count")
)
monthly_class["ym"] = monthly_class["year"].astype(str) + "-" + monthly_class["month"].astype(str).str.zfill(2)
monthly_class.sort_values("ym", inplace=True)

class_pivot = monthly_class.pivot_table(index="ym", columns="sep_class", values="count", fill_value=0)
class_pivot["involuntary_pct"] = (class_pivot["involuntary"] / class_pivot[["involuntary","voluntary","other"]].sum(axis=1) * 100).round(1)
class_pivot

sep_class,involuntary,other,voluntary,involuntary_pct
ym,,,,
2025-01,1857.0,1371.0,19062.0,8.3
2025-02,1729.0,1889.0,12017.0,11.1
2025-03,2945.0,1934.0,17235.0,13.3
2025-04,3462.0,1424.0,19290.0,14.3
2025-05,5014.0,1498.0,20224.0,18.8
2025-06,3576.0,1254.0,15286.0,17.8
2025-07,8258.0,1325.0,13384.0,36.0
2025-08,3782.0,1254.0,13493.0,20.4
2025-09,6043.0,1338.0,115669.0,4.9


## Agency Breakdown of Involuntary Separations

In [33]:
# Involuntary separations by agency and period
inv = df[df["sep_class"] == "involuntary"]

agency_inv = (
    inv.groupby(["agency_subelement_code", "agency_subelement", "period"])
    .size()
    .reset_index(name="count")
    .pivot_table(index=["agency_subelement_code", "agency_subelement"], columns="period", values="count", fill_value=0)
    .reset_index()
)

# Total involuntary per agency across all periods
agency_inv["total"] = agency_inv.select_dtypes("number").sum(axis=1)
agency_inv.sort_values("total", ascending=False).head(20)

period,agency_subelement_code,agency_subelement,2026 (DOGE year 2),Feb-Dec 2025 (DOGE year 1),Jan-2025 (transition),total
275,IN10,NATIONAL PARK SERVICE,836.0,4131.0,51.0,5018.0
370,VATA,VETERANS HEALTH ADMINISTRATION,349.0,4326.0,71.0,4746.0
52,AM00,U.S. AGENCY FOR INTERNATIONAL DEVELOPMENT,157.0,4007.0,2.0,4166.0
31,AFNG,AIR NATIONAL GUARD UNITS,319.0,2116.0,198.0,2633.0
38,AG11,FOREST SERVICE,79.0,2358.0,53.0,2490.0
76,ARNG,ARMY NATIONAL GUARD UNITS,305.0,1928.0,147.0,2380.0
236,HE36,FOOD AND DRUG ADMINISTRATION,112.0,1856.0,10.0,1978.0
238,HE38,NATIONAL INSTITUTES OF HEALTH,24.0,1878.0,21.0,1923.0
340,ST00,DEPARTMENT OF STATE,111.0,1477.0,32.0,1620.0
137,DD48,DEFENSE HUMAN RESOURCES ACTIVITY,1200.0,362.0,4.0,1566.0


In [34]:
# RIF vs Termination split by agency (top 20 by involuntary total)
top_agencies = agency_inv.sort_values("total", ascending=False).head(20)["agency_subelement_code"].tolist()

inv_type = (
    inv[inv["agency_subelement_code"].isin(top_agencies)]
    .groupby(["agency_subelement", "separation_category"])
    .size()
    .unstack(fill_value=0)
)
inv_type

separation_category,REDUCTION IN FORCE (RIF),TERMINATION (EXPIRED APPT/OTHER)
agency_subelement,,
AIR NATIONAL GUARD UNITS,0,2633
ANIMAL AND PLANT HEALTH INSPECTION SERVICE,0,635
ARMY NATIONAL GUARD UNITS,0,2380
BUREAU OF LAND MANAGEMENT,0,792
CENTERS FOR DISEASE CONTROL AND PREVENTION,573,664
DEFENSE HUMAN RESOURCES ACTIVITY,0,1566
DEPARTMENT OF DEFENSE EDUCATION ACTIVITY,0,871
DEPARTMENT OF STATE,898,722
FEDERAL AVIATION ADMINISTRATION,0,789


## Profile of Involuntary Separations
Who is being RIF'd and terminated — age, tenure, pay plan, appointment type.

In [35]:
# Compare profile: involuntary vs voluntary (shares)
profile_cols = {
    "age_bracket": "Age",
    "tenure": "Tenure",
    "pay_plan": "Pay Plan",
    "appointment_type": "Appointment Type",
    "work_schedule": "Work Schedule",
    "education_level_bracket": "Education",
}

for col, label in profile_cols.items():
    if col not in df.columns:
        continue
    tbl = (
        df.groupby([col, "sep_class"])
        .size()
        .unstack(fill_value=0)
    )
    # Normalize to % within each class
    tbl_pct = (tbl.div(tbl.sum(axis=0), axis=1) * 100).round(1)
    if "involuntary" in tbl_pct.columns and "voluntary" in tbl_pct.columns:
        tbl_pct["inv_minus_vol"] = tbl_pct["involuntary"] - tbl_pct["voluntary"]
        print(f"\n=== {label} ===")
        print(tbl_pct[["involuntary", "voluntary", "inv_minus_vol"]].sort_values("inv_minus_vol", ascending=False).to_string())


=== Age ===
sep_class     involuntary  voluntary  inv_minus_vol
age_bracket                                        
20-24                12.3        3.5            8.8
25-29                14.1        7.0            7.1
30-34                11.9        8.3            3.6
40-44                10.8        8.5            2.3
45-49                 9.4        7.3            2.1
35-39                10.6        8.9            1.7
LESS THAN 20          1.0        0.1            0.9
50-54                 7.8        9.6           -1.8
65 OR MORE            9.5       15.0           -5.5
55-59                 7.0       14.4           -7.4
60-64                 5.7       17.4          -11.7

=== Tenure ===
sep_class                                                                                                       involuntary  voluntary  inv_minus_vol
tenure                                                                                                                                           